<span style='font-size:large'>**Preprocessing**</span>

<span style='font-size:small'>This notebook focuses on the preprocessing stage of our capstone project. Here, we transform the cleaned text into a format suitable for TF\-IDF and structural topic modelling. The steps include tokenisation and removing stopwords. By completing this steps, we prepare the dataset for topic modelling and other text analyses in a structured and efficient way.</span>

<span style='font-size:medium'>**Relevant lecture:** </span>

<span style='font-size:small'>Course 2, Module 13: "Text Analysis for Environmental Data " by Luke Sanford — Covers core techniques for analysing environmental text—including cleaning, tokenisation, stop\-word removal, word\-frequency analysis, sentiment analysis, and building custom dictionaries.</span>

<span style='font-size:small'>Course 3, Module 13: "Converting Text to Quantitative Data" by Luke Sanford —explains how to transform raw text into numerical data using different preprocessing methods.</span>



In [3]:
#load required libraries
library(tidyverse)     # Data manipulation and visualization
library(tidytext)      # Text mining with tidy data principles
library(topicmodels)   # Topic modeling (LDA)
library(SnowballC)     # Stemming
library(tm)            # Text mining utilities
library(ggplot2)       # Data visualization
library(dplyr)         # Data manipulation
library(stringr)       # String manipulation
library(janitor)       # standardization
library(textstem)      # textdata preprocessing

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Loading required package: NLP




Attaching package: ‘NLP’




The following object is masked from ‘package:ggplot2’:

    annotate




...

Loading required package: sylly



For information on available language packages for 'koRpus', run

  available.koRpus.lang()

and see ?install.koRpus.lang()





Attaching package: ‘koRpus’




The following object is masked from ‘package:tm’:

    readTagged




The following object is masked from ‘package:readr’:

    tokenize




In [7]:
# Load the clean dataset
clean_docs <- read_csv("data/cleaned/clean_docs.csv")

# since the documents talk about the same project we tried to remove common words that are likely to be reocurring across all documents. 
# We then created a custom data frame for those common words 

custom_stops <- data.frame(word = c("project", "projects", "nrt", "northern", "kenya",
  "kenyan", "conservancy", "conservancies", "community", "communities", "carbon",
  "credit", "credits", "offset", "offsets", "grassland", "rangeland", "rangelands",
  "nkcp", "nkrcp", "verra", "trust", "report", "reports", "section", "figure",
  "table", "data", "year", "years", "nrtã", "nrtâ", "ã", "â"))

# Tokenize documents and clean tokens

doc_tokens <- clean_docs %>%
  unnest_tokens(word, text) %>%
  mutate(word = str_replace_all(word, "[^a-zA-Z0-9]", "")) %>%
  mutate(word = lemmatize_words(word)) %>%
  anti_join(stop_words, by = "word") %>%
  anti_join(custom_stops, by = "word") %>%
  filter(!str_detect(word, "^\\d+$")) %>%   
  filter(nchar(word) > 2)

# Check dimensions
cat("Total tokens:", nrow(doc_tokens), "\n")
cat("Unique words:", n_distinct(doc_tokens$word), "\n")
cat("Number of documents:", n_distinct(doc_tokens$source), "\n")

# Check for empty/NA tokens left
cat("Empty strings:", sum(doc_tokens$word == ""), "\n")
cat("NA tokens:", sum(is.na(doc_tokens$word)), "\n")

# look at the top most frequent words
head(word_counts, 30)

# look at the least frequent words 
tail(word_counts, 30)

# Check token distribution per source/document
doc_tokens %>%
  count(source) %>%
  arrange(desc(n)) 

# save raw tokens
write_csv(doc_tokens,"data/processed/doc_tokens.csv")

# save precounted okens for frequency analysis and visualisation
word_counts <- doc_tokens %>%
    count(source,word,sort = TRUE)
write_csv(word_counts,"data/processed/word_counts.csv")


Rows: 5 Columns: 5


── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): doc_id, doc_name, source, text
dbl (1): year



ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Total tokens: 7463 


Unique words: 1982 


...

Empty strings: 0 


NA tokens: 0 


source,word,n
<chr>,<chr>,<int>
official,household,104
official,graze,103
official,benefit,63
Independent,land,54
official,plan,54
official,livestock,49
official,increase,46
official,survey,45
official,woman,37


source,word,n
<chr>,<chr>,<int>
official,successfully,1
official,summarize,1
official,swiss,1
official,tendency,1
official,thankyou,1
official,theft,1
official,theme,1
official,thesis,1
official,threat,1


source,n
<chr>,<int>
Independent,3887
official,3576
